# Parking Analysis — shared inference pipeline

This notebook uses the same parking model, RT-DETR vehicle detector and occupancy rules as the browser app. No manual layout is required. For the easiest workflow, run `python start_app.py` or double-click `START_APP.cmd`.

Install dependencies with `python -m pip install -r requirements.txt` and verify weights with `python setup_models.py --check` first.


In [ ]:
from pathlib import Path
import sys, json
import cv2
import matplotlib.pyplot as plt
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "parking_lite").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from the project or notebooks folder")
sys.path.insert(0, str(ROOT))
from parking_lite.engine import ParkingEngine
from parking_lite.session import read_photo
from parking_lite.core import draw_analysis
engine = ParkingEngine(ROOT)
print(json.dumps(engine.model_info(), indent=2))


## Choose a photo

Use a real parking photograph. `balanced` runs the main models. `accurate` also checks crops around predicted empty bays and may take longer.


In [ ]:
IMAGE_PATH = ROOT / "samples" / "parking photos" / "parking.jpg"
QUALITY = "balanced"
image = read_photo(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(IMAGE_PATH)
analysis = engine.analyze(image, quality=QUALITY)
print(json.dumps(analysis["counts"], indent=2))
print(analysis["note"])
annotated = draw_analysis(image, analysis, show_vehicles=False)
plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off");


## Save the result

Red is occupied, green is a model prediction of availability, yellow is unknown. Estimated rectangles do not precisely trace white lines. Review accuracy on labelled photos from the intended viewpoints.


In [ ]:
output = ROOT / "outputs"
output.mkdir(exist_ok=True)
cv2.imwrite(str(output / "notebook-analysis.jpg"), annotated)
(output / "notebook-analysis.json").write_text(json.dumps(analysis, indent=2), encoding="utf8")
print("Saved to", output)
